# Python (Intermediate) -- Prerequisite Review

**Who this is for:** You already know Python. You can write 100+ line scripts, use classes, and debug with tracebacks. This notebook is a quick refresher and self-assessment -- not a tutorial from scratch.

**Estimated time:** 30-45 minutes

**What it covers:**
- Functions (args, kwargs, defaults, return values)
- Classes (init, methods, inheritance, dunder methods)
- List/dict/set comprehensions
- File I/O and context managers
- Error handling (try/except, custom exceptions)
- Debugging with tracebacks

If any section feels unfamiliar (not just rusty), check the **References** at the bottom before the course starts.

---
## 1. Functions

Quick reminders:
- Positional vs keyword arguments
- `*args` collects extra positional args into a tuple; `**kwargs` collects extra keyword args into a dict
- Default values are evaluated **once** at function definition (mutable default trap)
- Functions are first-class objects -- you can pass them around, return them, store them in lists

In [ ]:
# Example: *args, **kwargs, and default values

def train_model(model_name, *layers, lr=0.001, **extra_config):
    """Simulate model training setup."""
    print(f"Model: {model_name}")
    print(f"Layers: {layers}")
    print(f"Learning rate: {lr}")
    print(f"Extra config: {extra_config}")
    return {"model": model_name, "lr": lr, **extra_config}

config = train_model("resnet", 64, 128, 256, lr=0.01, dropout=0.5, batch_size=32)
print(f"\nReturned config: {config}")

In [ ]:
# Example: The mutable default trap

# BAD -- the list is shared across all calls
def append_to_list_bad(item, target=[]):
    target.append(item)
    return target

print(append_to_list_bad(1))  # [1]
print(append_to_list_bad(2))  # [1, 2]  -- probably not what you wanted

# GOOD -- use None as sentinel
def append_to_list_good(item, target=None):
    if target is None:
        target = []
    target.append(item)
    return target

print(append_to_list_good(1))  # [1]
print(append_to_list_good(2))  # [2]

### Exercise 1.1

Write a function `describe_data(*arrays, precision=2)` that:
- Accepts any number of lists of numbers
- Returns a list of dicts, one per array, with keys `"mean"`, `"min"`, `"max"` (rounded to `precision` decimals)
- Do NOT use numpy -- use built-in functions only

```python
describe_data([1, 2, 3], [10, 20], precision=1)
# [{'mean': 2.0, 'min': 1, 'max': 3}, {'mean': 15.0, 'min': 10, 'max': 20}]
```

In [ ]:
# YOUR CODE HERE


### Exercise 1.2

Write a function `compose(*funcs)` that takes any number of single-argument functions and returns a new function that applies them right-to-left (mathematical composition).

```python
double = lambda x: x * 2
add_one = lambda x: x + 1
square = lambda x: x ** 2

f = compose(square, add_one, double)  # square(add_one(double(x)))
f(3)  # square(add_one(6)) = square(7) = 49
```

In [ ]:
# YOUR CODE HERE


---
## 2. Classes

Quick reminders:
- `__init__` sets up instance state; `self` is the instance reference
- Inheritance: `class Child(Parent)` -- use `super()` to call parent methods
- Dunder methods: `__repr__`, `__str__`, `__len__`, `__eq__`, `__lt__`, `__getitem__`, etc.
- `@property` for computed attributes with getter/setter syntax

In [ ]:
# Example: A practical class with dunders

class Dataset:
    def __init__(self, features, labels):
        assert len(features) == len(labels), "Length mismatch"
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

    def __repr__(self):
        return f"Dataset(n_samples={len(self)}, n_features={len(self.features[0])})"


ds = Dataset([[1, 2], [3, 4], [5, 6]], [0, 1, 0])
print(ds)            # Dataset(n_samples=3, n_features=2)
print(len(ds))       # 3
print(ds[1])         # ([3, 4], 1)

In [ ]:
# Example: Inheritance and super()

class Metric:
    def __init__(self, name):
        self.name = name
        self.values = []

    def update(self, value):
        self.values.append(value)

    def compute(self):
        raise NotImplementedError


class MeanMetric(Metric):
    def __init__(self, name):
        super().__init__(name)

    def compute(self):
        if not self.values:
            return 0.0
        return sum(self.values) / len(self.values)

    def __repr__(self):
        return f"{self.name}: {self.compute():.4f}"


loss = MeanMetric("loss")
for v in [0.9, 0.7, 0.5, 0.3]:
    loss.update(v)
print(loss)  # loss: 0.6000

### Exercise 2.1

Create a `Matrix` class that:
- Takes a list of lists in `__init__` and stores it
- Has a `shape` property returning `(rows, cols)`
- Implements `__repr__` showing `Matrix(rows x cols)`
- Implements `__add__` for element-wise addition of two matrices (raise `ValueError` on shape mismatch)
- Implements `__getitem__` so `m[i, j]` returns the element at row i, col j

```python
a = Matrix([[1, 2], [3, 4]])
b = Matrix([[5, 6], [7, 8]])
c = a + b
print(c[0, 1])  # 8
print(c.shape)   # (2, 2)
```

In [ ]:
# YOUR CODE HERE


---
## 3. Comprehensions

Quick reminders:
- List: `[expr for x in iterable if condition]`
- Dict: `{key_expr: val_expr for x in iterable}`
- Set: `{expr for x in iterable}`
- Nested: `[expr for x in outer for y in inner]` (left-to-right = outer-to-inner)
- Generator expressions: `(expr for x in iterable)` -- lazy, memory-efficient

In [ ]:
# Example: Various comprehensions in a data-processing context

raw_data = [
    {"name": "Alice", "score": 85, "passed": True},
    {"name": "Bob", "score": 42, "passed": False},
    {"name": "Carol", "score": 91, "passed": True},
    {"name": "Dave", "score": 67, "passed": True},
]

# Dict comprehension: name -> score for those who passed
passed_scores = {d["name"]: d["score"] for d in raw_data if d["passed"]}
print(passed_scores)  # {'Alice': 85, 'Carol': 91, 'Dave': 67}

# Set comprehension: unique score brackets (tens)
brackets = {d["score"] // 10 * 10 for d in raw_data}
print(brackets)  # {80, 40, 90, 60}

# Nested list comprehension: flatten a matrix
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat = [x for row in matrix for x in row]
print(flat)  # [1, 2, 3, 4, 5, 6, 7, 8, 9]

### Exercise 3.1

Given a list of strings representing file paths, use comprehensions to:
1. Create a dict mapping each unique file extension to the count of files with that extension
2. Create a list of just the filenames (without directory path) for `.py` files only

```python
paths = [
    "src/model.py", "src/train.py", "data/train.csv",
    "data/test.csv", "README.md", "src/utils.py", "config.yaml"
]
# ext_counts -> {'.py': 3, '.csv': 2, '.md': 1, '.yaml': 1}
# py_files -> ['model.py', 'train.py', 'utils.py']
```

Hint: `str.rsplit('/', 1)` and `str.rsplit('.', 1)` are useful here.

In [ ]:
# YOUR CODE HERE


---
## 4. File I/O and Context Managers

Quick reminders:
- Always use `with open(...)` -- it guarantees the file is closed even if an exception occurs
- Modes: `'r'` (read text), `'w'` (write/overwrite), `'a'` (append), `'rb'`/`'wb'` (binary)
- `encoding='utf-8'` -- always specify it explicitly
- `json`, `csv` modules for structured data
- `pathlib.Path` is the modern way to handle file paths

In [ ]:
# Example: Writing and reading structured data
import json
import csv
from pathlib import Path
import tempfile

tmpdir = Path(tempfile.mkdtemp())

# Write JSON
metrics = {"epoch": 10, "loss": 0.023, "accuracy": 0.971}
with open(tmpdir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# Read it back
with open(tmpdir / "metrics.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
print(f"Loaded: {loaded}")

# Write CSV
rows = [{"name": "model_a", "acc": 0.92}, {"name": "model_b", "acc": 0.95}]
with open(tmpdir / "results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "acc"])
    writer.writeheader()
    writer.writerows(rows)

# Read CSV back
with open(tmpdir / "results.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)

### Exercise 4.1

Write a function `merge_json_files(file_paths, output_path)` that:
- Reads each JSON file (each contains a dict)
- Merges them into a single dict (later files overwrite earlier keys)
- Writes the merged dict to `output_path`
- Returns the merged dict

Use `pathlib.Path` and context managers. Handle the case where a file does not exist by printing a warning and skipping it.

In [ ]:
# YOUR CODE HERE


---
## 5. Error Handling

Quick reminders:
- `try / except / else / finally` -- `else` runs only if no exception, `finally` always runs
- Catch specific exceptions, not bare `except:`
- `raise` to re-raise; `raise NewError(...) from original` for chaining
- Custom exceptions: just subclass `Exception` (or a more specific built-in)
- Common: `ValueError`, `TypeError`, `KeyError`, `IndexError`, `FileNotFoundError`, `RuntimeError`

In [ ]:
# Example: Structured error handling with custom exceptions

class ConfigError(Exception):
    """Raised when configuration is invalid."""
    pass


class MissingKeyError(ConfigError):
    """Raised when a required config key is missing."""
    def __init__(self, key):
        self.key = key
        super().__init__(f"Missing required config key: '{key}'")


def load_config(config_dict, required_keys):
    """Validate that all required keys are present."""
    for key in required_keys:
        if key not in config_dict:
            raise MissingKeyError(key)
    return config_dict


# Usage
try:
    cfg = load_config({"lr": 0.01, "epochs": 10}, ["lr", "epochs", "batch_size"])
except MissingKeyError as e:
    print(f"Config error: {e}")
    print(f"The missing key was: {e.key}")
except ConfigError as e:
    print(f"General config error: {e}")

In [ ]:
# Example: else and finally

def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None
    except TypeError as e:
        print(f"Type error: {e}")
        return None
    else:
        # Only runs if no exception
        print(f"{a} / {b} = {result}")
        return result
    finally:
        # Always runs
        print("Division attempted.")

safe_divide(10, 3)
print()
safe_divide(10, 0)

### Exercise 5.1

Write a function `safe_convert(values, target_type)` that:
- Takes a list of values and a target type (e.g., `int`, `float`)
- Returns a list of converted values
- For values that fail conversion, insert `None` instead and print a warning with the index and original value
- Raise `TypeError` if `target_type` is not callable

```python
safe_convert(["1", "2.5", "abc", "4"], int)
# Warning: could not convert index 1 value '2.5' to <class 'int'>
# Warning: could not convert index 2 value 'abc' to <class 'int'>
# [1, None, None, 4]
```

In [ ]:
# YOUR CODE HERE


---
## 6. Debugging with Tracebacks

Quick reminders:
- Read tracebacks **bottom-up**: the last line is the exception, the lines above show the call stack
- Each frame shows: file, line number, function name, and the line of code
- Common patterns:
  - `AttributeError: 'NoneType' object has no attribute 'X'` -- something returned `None` unexpectedly
  - `IndexError: list index out of range` -- off-by-one or empty list
  - `KeyError: 'X'` -- missing dict key, often a typo or wrong data
  - `TypeError: func() got an unexpected keyword argument` -- API mismatch
- Use `traceback` module to capture/format tracebacks programmatically
- `python -m pdb script.py` or `breakpoint()` for interactive debugging

In [ ]:
# Example: Reading a traceback
# Run this cell and then read the traceback bottom-up.

def load_data(path):
    data = {"train": [1, 2, 3], "test": [4, 5]}
    return data.get(path)  # returns None if key not found

def preprocess(data):
    return [x * 2 for x in data]  # will fail if data is None

def pipeline(split):
    raw = load_data(split)
    processed = preprocess(raw)
    return processed

# This will raise an error -- read the traceback carefully
try:
    pipeline("validation")  # "validation" is not a valid key
except TypeError as e:
    import traceback
    traceback.print_exc()
    print("\n--- Diagnosis ---")
    print("The traceback tells us:")
    print("1. pipeline() called load_data('validation')")
    print("2. load_data returned None (key not found)")
    print("3. preprocess(None) tried to iterate over None")
    print("Fix: validate the return value of load_data, or use data['path'] to get a KeyError early.")

### Exercise 6.1

The following code has a bug. Run it, read the traceback, and fix the bug. Add a comment explaining what went wrong.

```python
class Tokenizer:
    def __init__(self, vocab):
        self.word_to_id = {w: i for i, w in enumerate(vocab)}
    
    def encode(self, text):
        tokens = text.lower().split()
        return [self.word_to_id[t] for t in tokens]

vocab = ["the", "cat", "sat", "on", "mat"]
tok = Tokenizer(vocab)
result = tok.encode("The cat sat on the mat")
print(result)
```

In [ ]:
# YOUR CODE HERE -- paste the code above, run it, read the traceback, then fix it


### Exercise 6.2

The function below silently produces wrong results (no exception raised). Find the bug by adding `print()` statements or using `breakpoint()`. Fix it and add a comment.

```python
def moving_average(values, window=3):
    """Compute the moving average with the given window size."""
    result = []
    for i in range(len(values) - window + 1):
        avg = sum(values[i:i + window] / window)
        result.append(avg)
    return result

# Expected: [2.0, 3.0, 4.0]
print(moving_average([1, 2, 3, 4, 5]))
```

In [ ]:
# YOUR CODE HERE -- paste the code above, find the bug, fix it


---
## Self-Assessment

You should be able to answer these questions confidently. If you cannot, review the references below before the course begins.

1. **What happens if you use a mutable object (like a list) as a default argument in a function definition? Why is this a problem?**

2. **Explain what `super().__init__()` does and when you need to call it. What happens if you forget?**

3. **Given the traceback below, identify the root cause:**
   ```
   Traceback (most recent call last):
     File "train.py", line 45, in main
       metrics = evaluate(model, test_loader)
     File "train.py", line 32, in evaluate
       predictions = model.predict(batch)
     File "model.py", line 18, in predict
       return self.weights @ inputs.T
   AttributeError: 'NoneType' object has no attribute 'T'
   ```
   What is `None` here, and where should you look to fix it?

4. **What is the difference between `except Exception as e` and a bare `except:`? Why does the bare version cause problems?**

---
## References

If any section above felt unfamiliar, work through these before the course starts:

- [Python Tutorial (official docs)](https://docs.python.org/3/tutorial/) -- Sections 4 (functions), 9 (classes), 8 (errors & exceptions)
- [Real Python -- Python Classes and OOP](https://realpython.com/python3-object-oriented-programming/) -- Solid walkthrough with practical examples
- [Automate the Boring Stuff, Ch. 8-11](https://automatetheboringstuff.com/) -- File I/O, debugging, and practical scripting patterns